In [3]:
import lightgbm as lgb

from lightgbm import LGBMClassifier

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix
)
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
import optuna

In [4]:
train_df = pd.read_csv("training_kerala(2003-2023).csv")
test_df = pd.read_csv("test_kerala(2024-25).csv")

print(train_df.shape)
print(test_df.shape)

(5774701, 14)
(352940, 14)


In [5]:
FEATURES = [
    "CHLOR_A",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos",
    "LAT_scaled",
    "LON_scaled"
]

TARGET = "PHYTOBLOOM"

In [6]:
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

In [7]:
lgb_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    random_state=42,
)

lgb_model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.056662 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 907
[LightGBM] [Info] Number of data points in the train set: 5774701, number of used features: 7
[LightGBM] [Info] Start training from score -0.070857
[LightGBM] [Info] Start training from score -3.417544
[LightGBM] [Info] Start training from score -3.335066


,objective,'multiclass'
,random_state,42
,num_class,3
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,class_weight,None
,min_split_gain,0.0


In [8]:
train_pred = lgb_model.predict(X_train)
test_pred = lgb_model.predict(X_test)

In [9]:
print("TRAIN RESULTS\n")

print(classification_report(y_train, train_pred))


print("Macro F1:",
      f1_score(y_train, train_pred, average='macro'))


print("\n\nTEST RESULTS\n")

print(classification_report(y_test, test_pred))


print("Macro F1:",
      f1_score(y_test, test_pred, average='macro'))

TRAIN RESULTS

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00   5379682
         1.0       0.86      0.88      0.87    189369
         2.0       0.86      0.85      0.85    205650

    accuracy                           0.99   5774701
   macro avg       0.90      0.91      0.91   5774701
weighted avg       0.99      0.99      0.99   5774701

Macro F1: 0.9054280101796729


TEST RESULTS

              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99    329160
         1.0       0.77      0.88      0.82     12313
         2.0       0.89      0.72      0.79     11467

    accuracy                           0.98    352940
   macro avg       0.88      0.86      0.87    352940
weighted avg       0.98      0.98      0.98    352940

Macro F1: 0.8698375255164669


In [10]:
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": lgb_model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance)

      Feature  Importance
5  LAT_scaled        2019
6  LON_scaled        1920
1     day_sin        1887
0     CHLOR_A        1718
2     day_cos        1259
3   month_sin         140
4   month_cos          57


In [12]:
def objective_multi_lgbm_v3(trial):
    params = {
        "objective": "multiclass",
        "num_class": 3,
        "metric": "multi_logloss",

        # Tightened further — trial 3 (best gap+F1) used 75; keep search
        # centered there instead of allowing drift back toward 250
        "n_estimators": trial.suggest_int("n_estimators", 20, 150, step=5),

        # Nudged down to match the lower n_estimators ceiling
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.08, log=True),

        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "num_leaves": trial.suggest_int("num_leaves", 30, 140),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 65),

        "subsample": trial.suggest_float("subsample", 0.65, 0.90),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.75, 1.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 1e-3, 0.10, log=True),

        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gaps, val_scores = [], []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr)

        train_f1 = f1_score(y_tr, model.predict(X_tr), average="macro")
        val_f1 = f1_score(y_val, model.predict(X_val), average="macro")

        val_scores.append(val_f1)
        gaps.append(train_f1 - val_f1)

    return np.mean(val_scores), np.mean(gaps)

In [13]:
study_lgbm_v3 = optuna.create_study(directions=["maximize", "minimize"])
study_lgbm_v3.optimize(objective_multi_lgbm_v3, n_trials=25, show_progress_bar=True)

[I 2026-07-12 02:14:40,754] A new study created in memory with name: no-name-8ea932d7-2d47-4ea7-a4fb-d6b400f31af9


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2026-07-12 02:15:35,060] Trial 0 finished with values: [0.8873597213454977, 9.258847642656942e-05] and parameters: {'n_estimators': 30, 'learning_rate': 0.06870143886404399, 'max_depth': 5, 'num_leaves': 58, 'min_child_samples': 45, 'subsample': 0.7197802305629011, 'colsample_bytree': 0.8894962193153785, 'reg_alpha': 0.5635902917290259, 'reg_lambda': 0.30521896147818905, 'min_split_gain': 0.0021717037181374806}.
[I 2026-07-12 02:18:19,238] Trial 1 finished with values: [0.9026867160188736, 0.0014319785444117717] and parameters: {'n_estimators': 95, 'learning_rate': 0.03761746662648442, 'max_depth': 9, 'num_leaves': 105, 'min_child_samples': 31, 'subsample': 0.777379843684652, 'colsample_bytree': 0.7742774223422424, 'reg_alpha': 0.004782872436877684, 'reg_lambda': 0.050767406902628445, 'min_split_gain': 0.03940582055465489}.
[I 2026-07-12 02:21:10,442] Trial 2 finished with values: [0.9092201274393954, 0.002899491380647978] and parameters: {'n_estimators': 105, 'learning_rate': 0.047

In [20]:
optuna.visualization.plot_param_importances(study_lgbm_v3)

In [15]:
best_trials = study_lgbm_v3.best_trials
i=0
for t in best_trials:
    print(f"{i} val_f1={t.values[0]:.4f}, gap={t.values[1]:.4f}, params={t.params}")
    i+=1

0 val_f1=0.8874, gap=0.0001, params={'n_estimators': 30, 'learning_rate': 0.06870143886404399, 'max_depth': 5, 'num_leaves': 58, 'min_child_samples': 45, 'subsample': 0.7197802305629011, 'colsample_bytree': 0.8894962193153785, 'reg_alpha': 0.5635902917290259, 'reg_lambda': 0.30521896147818905, 'min_split_gain': 0.0021717037181374806}
1 val_f1=0.9092, gap=0.0029, params={'n_estimators': 105, 'learning_rate': 0.04779227348113779, 'max_depth': 10, 'num_leaves': 134, 'min_child_samples': 41, 'subsample': 0.6777435694494576, 'colsample_bytree': 0.9277544750115896, 'reg_alpha': 0.09094007911892434, 'reg_lambda': 0.0019433117538520286, 'min_split_gain': 0.004429607743759151}
2 val_f1=0.8910, gap=0.0004, params={'n_estimators': 65, 'learning_rate': 0.03493666974044454, 'max_depth': 6, 'num_leaves': 109, 'min_child_samples': 63, 'subsample': 0.7496389197715401, 'colsample_bytree': 0.8993287210472342, 'reg_alpha': 0.5866014018941246, 'reg_lambda': 0.31057600374367395, 'min_split_gain': 0.0127487

In [16]:
i=0
for t in best_trials:
    params=t.params
    lgb_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    **params,
    random_state=42
    )

    lgb_model.fit(X_train, y_train)
    y_pred1 = lgb_model.predict(
        X_train
    )

    y_pred = lgb_model.predict(
        X_test
    )
    macro_f1_train = f1_score(
        y_train,
        y_pred1,
        average="macro"
    )

    macro_f1_test = f1_score(
        y_test,
        y_pred,
        average="macro"
    )
    gap=macro_f1_train-macro_f1_test
    print(f"{i} Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")
    i+=1


0 Gap: 0.023570464072627306 Train Macro F1 : 0.8868985556702104  Test Macro F1 : 0.8633280915975831
1 Gap: 0.04404834216402975 Train Macro F1 : 0.9113837117027678  Test Macro F1 : 0.867335369538738
2 Gap: 0.0250463822936412 Train Macro F1 : 0.891158300371572  Test Macro F1 : 0.8661119180779308
3 Gap: 0.040569920523730874 Train Macro F1 : 0.9088167860706814  Test Macro F1 : 0.8682468655469505
4 Gap: 0.050142153297917824 Train Macro F1 : 0.917182693911014  Test Macro F1 : 0.8670405406130962
5 Gap: 0.038238885690556 Train Macro F1 : 0.9063113618501076  Test Macro F1 : 0.8680724761595516
6 Gap: 0.03155922866173477 Train Macro F1 : 0.9007669704367077  Test Macro F1 : 0.869207741774973
7 Gap: 0.03630452369184056 Train Macro F1 : 0.9046230070621348  Test Macro F1 : 0.8683184833702943
8 Gap: 0.028769223338229843 Train Macro F1 : 0.8977528365625017  Test Macro F1 : 0.8689836132242719
9 Gap: 0.02347333145591257 Train Macro F1 : 0.8819985254135752  Test Macro F1 : 0.8585251939576626
10 Gap: 0.023

In [19]:
params=best_trials[10].params
params

{'n_estimators': 130,
 'learning_rate': 0.043381244467624716,
 'max_depth': 4,
 'num_leaves': 109,
 'min_child_samples': 33,
 'subsample': 0.8917053192257713,
 'colsample_bytree': 0.9642181939482745,
 'reg_alpha': 0.044049787119522996,
 'reg_lambda': 0.05299143759204272,
 'min_split_gain': 0.0876499877099334}

In [2]:
params={'n_estimators': 130,
 'learning_rate': 0.043381244467624716,
 'max_depth': 4,
 'num_leaves': 109,
 'min_child_samples': 33,
 'subsample': 0.8917053192257713,
 'colsample_bytree': 0.9642181939482745,
 'reg_alpha': 0.044049787119522996,
 'reg_lambda': 0.05299143759204272,
 'min_split_gain': 0.0876499877099334}

In [7]:
lgb_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    **params,
    random_state=42
    )

lgb_model.fit(X_train, y_train)
y_pred1 = lgb_model.predict(
    X_train
)

y_pred = lgb_model.predict(
    X_test
)
macro_f1_train = f1_score(
    y_train,
    y_pred1,
    average="macro"
)

macro_f1_test = f1_score(
    y_test,
    y_pred,
    average="macro"
)
gap=macro_f1_train-macro_f1_test
print(f" Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.036181 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 907
[LightGBM] [Info] Number of data points in the train set: 5774701, number of used features: 7
[LightGBM] [Info] Start training from score -0.070857
[LightGBM] [Info] Start training from score -3.417544
[LightGBM] [Info] Start training from score -3.335066
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

In [8]:
import joblib

joblib.dump(lgb_model, 'lgb_model_Kerala.pkl')

['lgb_model_Kerala.pkl']